Running importance analysis with Python API
=====================================

This is an *VariantSpark* example notebook.


One of the main applications of VariantSpark is discovery of genomic variants correlated with a response variable (e.g. case vs control) using random forest gini importance.

The `chr22_1000.vcf` is a very small sample of the chromosome 22 VCF file from the 1000 Genomes Project.

`chr22-labels.csv` is a CSV file with sample response variables (labels). In fact the labels directly represent the number of alternative alleles for each sample at a specific genomic position. E.g.: column 22_16050408 has labels derived from variants in chromosome 22 position 16050408. We would expect then that position 22:16050408 in the VCF file is strongly correlated with the label 22_16050408.

Both data sets are located in the `..\data` directory.

This notebook demonstrates how to run importance analysis on these data with *VariantSpark* Python API.

Step 1: Create a spark session with VariantSpark jar attached.

In [3]:
import varspark as vs
from pyspark.sql import SparkSession 
spark = SparkSession.builder.config('spark.jars', vs.find_jar()).getOrCreate()

Step 2: Create a `VarsparkContext` using `SparkSession` object (here injected as `spark`):

In [4]:
vc = vs.VarsparkContext(spark, silent = True)

Step 3: Load the features `fs` and labels `ls` from data files.

In [ ]:
fs = vc.import_vcf('../../data/chr22_1000.vcf')
ls = vc.load_label('../../data/chr22-labels.csv', '22_16050408')

../../data/chr22_1000.vcf is loading to spark RDD, isBGZFile: false


Step 4: Fit an RF model to the data.

In [8]:
rf = vs.RandomForestModel(vc, mtry_fraction=0.10, min_node_size=5, max_depth=10, seed=13)
rf.fit_trees(fs, ls, n_trees=500, batch_size=20)

Step 5: Retrieve top important variables from the trained model:

In [21]:
ia = rf.importance_analysis()
top_variables = ia.important_variables(limit=10, normalized=True)

Step 6: Display the results.

In [ ]:
print("%s\t%s" % ('Variable', 'Importance'))
for var_and_imp in top_variables.values:
    print("%s\t%s" % tuple(var_and_imp))

Variable	Importance
22_16050408_T_C	0.0018769752144287713
22_16050678_C_T	0.0012120864823132948
22_16051107_C_A	0.000960270540328608
22_16051480_T_C	0.0009479044142555816
22_16052838_T_A	0.0006796228627158183
22_16053197_G_T	0.000570540351756131
22_16051882_C_T	0.0004550547173357528
22_16053797_T_C	0.0004161458343297709
22_16053435_G_T	0.000386837256156286
22_16053727_T_G	0.0003728045862310142


For more information on using *VariantSpark* and the Python API please visit the [documentation](http://variantspark.readthedocs.io/en/latest/).